# Demodulate Packed Audio Pipeline

This notebook runs the one-DMA packed AXI-Stream version of the project.

## Pipeline
1. Load a `.wav` file
2. Convert to mono and normalize
3. Generate a carrier in software
4. Modulate in software
5. Pack `(modulated_sample, carrier_sample)` into one 64-bit AXI-Stream word
6. Send packed data to the FPGA
7. FPGA unpacks and multiplies the values
8. Return the multiplied result to software
9. Apply low-pass filtering in software
10. Save recovered hardware and software `.wav` outputs

## Hardware assumptions
Your overlay should contain:
- `axi_dma`
- `demodulate_packed_0`

If Vivado exported different names, update them in the configuration cell.

In [ ]:
import numpy as np
import time
from pynq import Overlay, allocate
from scipy.io import wavfile

## Configuration

In [ ]:
N = 1024
BITSTREAM_PATH = "/home/xilinx/jupyter_notebooks/final project/demodulate_packed.bit"
WAV_PATH = "/home/xilinx/jupyter_notebooks/final project/input.wav"

DMA_NAME = "axi_dma"
IP_NAME = "demodulate_packed_0"

CARRIER_FREQ_HZ = 4000
LPF_CUTOFF_HZ = 2000

PRINT_IP_INFO = True

## Helper functions

In [ ]:
def pad_or_trim(x, n):
    x = np.asarray(x, dtype=np.float32)
    if len(x) < n:
        x = np.pad(x, (0, n - len(x)), mode="constant")
    else:
        x = x[:n]
    return x.astype(np.float32)

def normalize_audio(x):
    x = np.asarray(x, dtype=np.float32)
    peak = np.max(np.abs(x))
    if peak == 0:
        return x
    return x / peak

def pack_signal_and_carrier(signal_f32, carrier_f32):
    signal_f32 = np.asarray(signal_f32, dtype=np.float32)
    carrier_f32 = np.asarray(carrier_f32, dtype=np.float32)

    sig_u32 = signal_f32.view(np.uint32)
    car_u32 = carrier_f32.view(np.uint32)

    packed_u64 = (car_u32.astype(np.uint64) << 32) | sig_u32.astype(np.uint64)
    return packed_u64

def lowpass_fft(signal, sample_rate, cutoff_hz):
    spectrum = np.fft.rfft(signal)
    freqs = np.fft.rfftfreq(len(signal), d=1 / sample_rate)
    spectrum[freqs > cutoff_hz] = 0
    filtered = np.fft.irfft(spectrum, n=len(signal))
    return filtered.astype(np.float32)

## Load overlay

In [ ]:
ol = Overlay(BITSTREAM_PATH)

if PRINT_IP_INFO:
    print("IP blocks:", list(ol.ip_dict.keys()))

dma = getattr(ol, DMA_NAME)
ip = getattr(ol, IP_NAME)

## Load audio and generate carrier

In [ ]:
rate, audio = wavfile.read(WAV_PATH)

if audio.ndim == 2:
    audio = audio.mean(axis=1)

audio = audio.astype(np.float32)

if np.max(np.abs(audio)) > 0:
    audio = audio / np.max(np.abs(audio))

audio_chunk = pad_or_trim(audio, N)

t = np.arange(N) / rate
carrier = np.cos(2 * np.pi * CARRIER_FREQ_HZ * t).astype(np.float32)

print("Sample rate:", rate)
print("Audio chunk shape:", audio_chunk.shape)
print("Carrier shape:", carrier.shape)
print("First 10 original samples:", audio_chunk[:10])

## Modulate in software

In [ ]:
modulated = audio_chunk * carrier

print("First 10 modulated samples:", modulated[:10])

## Pack modulated signal and carrier

In [ ]:
packed_input = pack_signal_and_carrier(modulated, carrier)

print("Packed input dtype:", packed_input.dtype)
print("Packed input shape:", packed_input.shape)
print("First 5 packed words:", packed_input[:5])

## Allocate DMA buffers

In [ ]:
in_buffer = allocate(shape=(N,), dtype=np.uint64)
out_buffer = allocate(shape=(N,), dtype=np.float32)

in_buffer[:] = packed_input
out_buffer[:] = 0.0

in_buffer.flush()
out_buffer.flush()

## Run FPGA multiply

In [ ]:
print("Starting hardware execution...")
start_hw = time.perf_counter()

dma.recvchannel.transfer(out_buffer)
ip.write(0x00, 0x01)
dma.sendchannel.transfer(in_buffer)

dma.sendchannel.wait()
dma.recvchannel.wait()

end_hw = time.perf_counter()

out_buffer.invalidate()
mixed_back_hw = np.copy(out_buffer)

print(f"Hardware finished successfully in {end_hw - start_hw:.6f} seconds")
print("First 10 mixed_back_hw samples:", mixed_back_hw[:10])

## Software reference for multiply

In [ ]:
mixed_back_sw = modulated * carrier

mse_mult = np.mean((mixed_back_sw - mixed_back_hw) ** 2)
max_err_mult = np.max(np.abs(mixed_back_sw - mixed_back_hw))

print("First 10 mixed_back_sw samples:", mixed_back_sw[:10])
print("Multiply-stage MSE:", mse_mult)
print("Multiply-stage Max err:", max_err_mult)

## Low-pass filter and recover audio

In [ ]:
demod_hw = 2.0 * lowpass_fft(mixed_back_hw, rate, LPF_CUTOFF_HZ)
demod_sw = 2.0 * lowpass_fft(mixed_back_sw, rate, LPF_CUTOFF_HZ)

demod_hw = normalize_audio(demod_hw)
demod_sw = normalize_audio(demod_sw)
audio_ref = normalize_audio(audio_chunk)

print("First 10 original audio samples:", audio_ref[:10])
print("First 10 recovered HW samples:", demod_hw[:10])
print("First 10 recovered SW samples:", demod_sw[:10])

## Compare recovered signals

In [ ]:
mse_recovered = np.mean((demod_hw - demod_sw) ** 2)
max_err_recovered = np.max(np.abs(demod_hw - demod_sw))

print("Recovered-signal MSE:", mse_recovered)
print("Recovered-signal Max err:", max_err_recovered)

## Save output WAV files

In [ ]:
wavfile.write(
    "/home/xilinx/jupyter_notebooks/final project/recovered_hw.wav",
    rate,
    np.int16(np.clip(demod_hw, -1.0, 1.0) * 32767)
)

wavfile.write(
    "/home/xilinx/jupyter_notebooks/final project/recovered_sw.wav",
    rate,
    np.int16(np.clip(demod_sw, -1.0, 1.0) * 32767)
)

print("Saved:")
print("/home/xilinx/jupyter_notebooks/final project/recovered_hw.wav")
print("/home/xilinx/jupyter_notebooks/final project/recovered_sw.wav")

## Cleanup

In [ ]:
in_buffer.close()
out_buffer.close()
print("Buffers closed.")